In [ ]:
# 1. Install Unsloth first
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Install dependencies.
# We removed the specific 'xformers<0.0.27' constraint that was causing the build error.
# We also install xformers separately to ensure it grabs the correct binary wheel.
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install xformers

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-78r076ly/unsloth_87154870cf1740fb8464a4bd2e61e575
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-78r076ly/unsloth_87154870cf1740fb8464a4bd2e61e575
  Resolved https://github.com/unslothai/unsloth.git to commit 33b0343ec56595d4e7d7cdd25f173207ebf991b0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.2/181.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 82.8 MB/s eta 0:00:00
 

In [ ]:
# Fix the trl version mismatch required by Unsloth
!pip install "trl>=0.18.2,<=0.24.0"

  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.8.6
    Uninstalling trl-0.8.6:
      Successfully uninstalled trl-0.8.6


In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import os

# 1. CONFIGURATION
max_seq_length = 2048
dtype = None
load_in_4bit = True

# 2. LOAD MODEL
print("Loading Phi-3.5 model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-3.5-mini-instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 3. ADD LoRA ADAPTERS
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

# 4. PREPARE YOUR DATA
# In Colab, uploaded files sit in the default folder or '/content/'
input_file = "clean_training_data.txt"

# Verify file exists before crashing
if not os.path.exists(input_file):
    raise FileNotFoundError(f"Could not find {input_file}. Did you upload it to the Files sidebar?")

def prepare_dataset(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    # Split into chunks
    chunk_size = 2000
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

    return Dataset.from_dict({"text": chunks})

dataset = prepare_dataset(input_file)
print(f"Dataset prepared with {len(dataset)} chunks.")

# 5. TRAIN
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",

    ),
)

print("Starting training...")
trainer_stats = trainer.train()

# 6. INFERENCE TEST
FastLanguageModel.for_inference(model)
inputs = tokenizer(
    [
        "User: Explain the concept of Pipelining in computer architecture.\nAssistant: "
    ], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
print(tokenizer.batch_decode(outputs))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Phi-3.5 model...
==((====))==  Unsloth 2026.1.2: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss
1,2.754700
2,2.662500
3,2.547700
4,2.744200
5,2.814400
6,2.670300
7,2.623900
8,2.667300
9,2.570500
10,2.479200


wandb: WARNING URL not available in offline run


train/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/global_step,▁▁▁▁▁▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇███
train/grad_norm,▁▁▁▁▁▁▂█▂▂▂▂▂▂▂▂▂▂▂▂▂▃▃▃▂▃▃▃▃▃▃▄▃▃▃▄▄▃▃▃
train/learning_rate,▁▂▄▇███▇▇▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁
train/loss,▇██▇▇▆▆▇▅▅▆▄▅▅▄▄▅▄▄▃▃▂▃▂▄▂▂▂▂▂▃▂▂▁▁▁▂▂▁▁
total_flos,5824348479590400.0
train/epoch,7.5
train/global_step,60
train/grad_norm,0.39433
train/learning_rate,0.0
train/loss,1.8666


['User: Explain the concept of Pipelining in computer architecture.\nAssistant:  Pipelining is a technique used in computer architecture to improve the performance of a processor. It is based on the concept of an assembly line, where each worker performs a small part of the overall task, and the output of one worker is the input of the next. In the case of a computer, the workers are functional units, and the task is to execute a sequence of instructions. Pipelining allows multiple instructions to be executed simultaneously by overlapping the execution of instructions. For example, if the processor takes 2 clock cycles to execute an instruction, then the processor can execute two instructions in every clock cycle, for']


In [ ]:
import torch # Ensure torch is imported

# 1. Fix the missing pad token issue globally
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

FastLanguageModel.for_inference(model)

print("Bot is ready! Type 'exit' to stop.")
print("-" * 30)

while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        break

    messages = [
        {"role": "user", "content": user_input}
    ]

    # Generate the input IDs
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    # Create an attention mask (1 for real data, 0 for padding)
    # Since we are doing 1-by-1 chat, everything is real data, so we use ones_like
    attention_mask = torch.ones_like(input_ids)

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,  # <--- This fixes the warning
        pad_token_id=tokenizer.pad_token_id, # <--- Explicitly sets the pad token
        max_new_tokens=256, # we need to change this to make the response bigger and shorter for the time being , and we should think of something dynamic
        use_cache=True,
        temperature=0.3,
    )

    decoded_output = tokenizer.batch_decode(outputs)[0]

    # Clean up output
    try:
        response = decoded_output.split("<|assistant|>")[1].replace("<|end|>", "")
    except IndexError:
        response = decoded_output

    print(f"Bot: {response.strip()}")
    print("-" * 30)

Bot is ready! Type 'exit' to stop.
------------------------------
You: what is Computer Architecture 
Bot: Computer architecture refers to the design and structure of a computer system. It encompasses the hardware and the interface between the hardware and the software. Computer architecture is concerned with the design of the components of a computer system, the interconnections between those components, and the description of the interface between the hardware and the software.

Computer architecture can be divided into several sub-areas:

1. Instruction set architecture (ISA): the interface between the hardware and the high-level language programs. It defines the instruction set, the data types, the addressing modes, and the memory architecture.

2. Microarchitecture (or computer organization): the implementation of the ISA. It defines the datapath, the control unit, and the memory hierarchy.

3. Pipelining: the technique of overlapping the execution of instructions to increase thro

## 🤖 AI Agent with Tool Use
This agent wraps the fine-tuned Phi-3.5 model with 5 tools:
- **Python Sandbox** – executes Python code safely
- **Assembly Coder** – generates x86-64 NASM assembly
- **Relevance Verifier** – scores response relevance vs. query
- **Calculator** – safe math expression evaluator
- **RAG Tool** – retrieves context from a local vector store


In [ ]:
# ─── CELL: Install agent dependencies ───────────────────────────────────────
!pip install faiss-cpu sentence-transformers numpy --quiet


In [ ]:
# ─── CELL: Tool Definitions ─────────────────────────────────────────────────
import io
import re
import ast
import math
import operator
import contextlib
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Any

# ── 1. Python Sandbox ────────────────────────────────────────────────────────
SAFE_BUILTINS = {
    k: __builtins__[k] if isinstance(__builtins__, dict) else getattr(__builtins__, k)
    for k in ['print', 'range', 'len', 'enumerate', 'zip', 'map', 'filter',
               'list', 'dict', 'set', 'tuple', 'int', 'float', 'str', 'bool',
               'sum', 'min', 'max', 'abs', 'round', 'sorted', 'reversed',
               'type', 'isinstance', 'repr', 'None', 'True', 'False']
    if (isinstance(__builtins__, dict) and k in __builtins__)
       or (not isinstance(__builtins__, dict) and hasattr(__builtins__, k))
}
SAFE_BUILTINS['__import__'] = None  # block imports inside sandbox

def python_sandbox(code: str) -> str:
    """
    Executes Python code in a restricted namespace.
    Returns stdout output or error message.
    """
    buf = io.StringIO()
    ns = {'__builtins__': SAFE_BUILTINS, 'math': math, 'np': np}
    try:
        with contextlib.redirect_stdout(buf):
            exec(compile(code, '<sandbox>', 'exec'), ns)
        return buf.getvalue().strip() or '(no output)'
    except Exception as e:
        return f'[SandboxError] {type(e).__name__}: {e}'


# ── 2. Assembly Language Coder ───────────────────────────────────────────────
_ASM_TEMPLATES = {
    'add':  ('Adding two numbers',
             'section .text\nglobal _start\n_start:\n    mov eax, {a}\n    mov ebx, {b}\n    add eax, ebx      ; eax = {a} + {b}\n    ; result in eax\n'),
    'mul':  ('Multiplying two numbers',
             'section .text\nglobal _start\n_start:\n    mov eax, {a}\n    mov ebx, {b}\n    imul eax, ebx     ; eax = {a} * {b}\n    ; result in eax\n'),
    'loop': ('Loop n times',
             'section .text\nglobal _start\n_start:\n    mov ecx, {a}    ; loop counter\n.loop:\n    ; --- body ---\n    dec ecx\n    jnz .loop\n    ; done\n'),
}

def assembly_coder(task: str) -> str:
    """
    Generates x86-64 NASM assembly for simple arithmetic / loop tasks.
    task examples: 'add 3 5', 'mul 4 7', 'loop 10'
    """
    task = task.strip().lower()
    parts = task.split()
    op = parts[0] if parts else ''
    nums = [int(x) for x in parts[1:] if x.lstrip('-').isdigit()]
    a = nums[0] if len(nums) > 0 else 0
    b = nums[1] if len(nums) > 1 else 0
    if op in _ASM_TEMPLATES:
        desc, tmpl = _ASM_TEMPLATES[op]
        return f'; {desc}\n' + tmpl.format(a=a, b=b)
    # Fallback: generic snippet using the model
    return (
        f'; x86-64 NASM snippet for: {task}\n'
        'section .data\n'
        '    msg db "result", 10\n'
        'section .text\n'
        'global _start\n'
        '_start:\n'
        '    ; TODO: implement logic for: ' + task + '\n'
        '    mov eax, 60   ; sys_exit\n'
        '    xor edi, edi\n'
        '    syscall\n'
    )


# ── 3. Relevance Verifier ────────────────────────────────────────────────────
_embedder = SentenceTransformer('all-MiniLM-L6-v2')

def relevance_verifier(query: str, response: str, threshold: float = 0.35) -> Dict[str, Any]:
    """
    Scores semantic similarity between query and response.
    Returns dict with score, verdict ('relevant'|'irrelevant'), and explanation.
    """
    emb = _embedder.encode([query, response], normalize_embeddings=True)
    score = float(np.dot(emb[0], emb[1]))
    verdict = 'relevant' if score >= threshold else 'irrelevant'
    return {
        'score': round(score, 4),
        'verdict': verdict,
        'threshold': threshold,
        'explanation': f'
        Cosine similarity = {score:.4f} ({'≥' if score>=threshold else '<'} {threshold})'
    }


# ── 4. Calculator ────────────────────────────────────────────────────────────
_ALLOWED_NODES = (
    ast.Expression, ast.BinOp, ast.UnaryOp, ast.Num,
    ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Pow,
    ast.Mod, ast.FloorDiv, ast.USub, ast.UAdd,
    ast.Constant, ast.Call, ast.Name, ast.Load
)
_SAFE_NAMES = {k: getattr(math, k) for k in dir(math) if not k.startswith('_')}

def calculator(expression: str) -> str:
    """
    Safely evaluates a math expression string.
    Supports +, -, *, /, **, %, // and math.* functions.
    """
    try:
        tree = ast.parse(expression.strip(), mode='eval')
        for node in ast.walk(tree):
            if not isinstance(node, _ALLOWED_NODES):
                return f'[CalcError] Disallowed operation: {type(node).__name__}'
        result = eval(compile(tree, '<calc>', 'eval'), {'__builtins__': {}}, _SAFE_NAMES)
        return str(result)
    except Exception as e:
        return f'[CalcError] {e}'


# ── 5. RAG Tool ──────────────────────────────────────────────────────────────
class RAGTool:
    """
    Lightweight in-memory FAISS vector store for retrieval-augmented generation.
    Call .build(docs) once, then .retrieve(query, k) at inference time.
    """
    def __init__(self):
        self.docs: List[str] = []
        self.index = None
        self._embedder = _embedder  # reuse the same model

    def build(self, docs: List[str]) -> None:
        """Index a list of text chunks."""
        self.docs = docs
        embs = self._embedder.encode(docs, normalize_embeddings=True).astype('float32')
        dim = embs.shape[1]
        self.index = faiss.IndexFlatIP(dim)  # Inner-product = cosine for normalized vecs
        self.index.add(embs)
        print(f'[RAG] Indexed {len(docs)} documents.')

    def retrieve(self, query: str, k: int = 3) -> List[Dict]:
        """Return top-k most relevant chunks with scores."""
        if self.index is None:
            return [{'chunk': '(RAG not built yet)', 'score': 0.0}]
        q_emb = self._embedder.encode([query], normalize_embeddings=True).astype('float32')
        scores, indices = self.index.search(q_emb, k)
        return [
            {'chunk': self.docs[i], 'score': round(float(s), 4)}
            for s, i in zip(scores[0], indices[0]) if i >= 0
        ]

# Build RAG index from the same training file used for fine-tuning
rag = RAGTool()
try:
    with open('clean_training_data.txt', 'r', encoding='utf-8') as f:
        raw = f.read()
    chunks = [raw[i:i+500] for i in range(0, len(raw), 500) if raw[i:i+500].strip()]
    rag.build(chunks)
except FileNotFoundError:
    print('[RAG] clean_training_data.txt not found — RAG will return empty results.')
    print('[RAG] Either upload the file or call rag.build(["doc1", "doc2", ...]) manually.')

print('✅ All tools ready: python_sandbox | assembly_coder | relevance_verifier | calculator | rag')


In [ ]:
# ─── CELL: Agent Core (Router + Chat Loop) ──────────────────────────────────
import re
import json

# ── Tool dispatcher ──────────────────────────────────────────────────────────
TOOL_PATTERNS = [
    # (regex on user query, tool_name, extractor_fn)
    (r'(?:run|execute|code|python)[:\s]+(.+)', 'python_sandbox',
        lambda m: m.group(1).strip()),
    (r'(?:assemble|asm|assembly)[:\s]+(.+)',   'assembly_coder',
        lambda m: m.group(1).strip()),
    (r'(?:calculate|compute|eval)[:\s]+(.+)',  'calculator',
        lambda m: m.group(1).strip()),
    (r'(?:search|retrieve|rag|lookup)[:\s]+(.+)', 'rag',
        lambda m: m.group(1).strip()),
]

def detect_tool(user_input: str):
    """
    Returns (tool_name, tool_arg) if a trigger keyword is found,
    otherwise (None, None) → fall through to the LLM.
    """
    for pattern, tool_name, extractor in TOOL_PATTERNS:
        m = re.search(pattern, user_input, re.IGNORECASE | re.DOTALL)
        if m:
            return tool_name, extractor(m)
    return None, None

def run_tool(tool_name: str, arg: str) -> str:
    if tool_name == 'python_sandbox':
        return python_sandbox(arg)
    elif tool_name == 'assembly_coder':
        return assembly_coder(arg)
    elif tool_name == 'calculator':
        return calculator(arg)
    elif tool_name == 'rag':
        hits = rag.retrieve(arg, k=3)
        return '\n---\n'.join(f'[score={h["score"]}] {h["chunk"]}' for h in hits)
    return '(unknown tool)'


# ── Model inference helper ────────────────────────────────────────────────────
def model_respond(prompt: str, rag_context: str = '') -> str:
    """
    Calls the fine-tuned Phi-3.5 model.
    Prepends RAG context to the system message when available.
    """
    import torch
    system = 'You are a helpful computer architecture assistant.'
    if rag_context:
        system += f'\n\nRelevant context:\n{rag_context}'

    messages = [
        {'role': 'system', 'content': system},
        {'role': 'user',   'content': prompt}
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    attention_mask = torch.ones_like(input_ids)

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        pad_token_id=tokenizer.pad_token_id,
        max_new_tokens=256,
        use_cache=True,
        temperature=0.3,
    )
    decoded = tokenizer.batch_decode(outputs)[0]
    try:
        return decoded.split('<|assistant|>')[1].replace('<|end|>', '').strip()
    except IndexError:
        return decoded.strip()


# ── Agent loop ────────────────────────────────────────────────────────────────
print('🤖 AI Agent ready! Tools: python, asm, calculator, rag, verifier')
print('Triggers: "run: <code>" | "asm: <task>" | "calculate: <expr>" | "search: <query>"')
print('Type "exit" to quit.\n' + '-' * 50)

while True:
    user_input = input('You: ').strip()
    if user_input.lower() in ('exit', 'quit'):
        break
    if not user_input:
        continue

    tool_name, tool_arg = detect_tool(user_input)

    if tool_name:
        print(f'[Agent] → Tool: {tool_name}')
        tool_result = run_tool(tool_name, tool_arg)
        print(f'[Tool Result]\n{tool_result}')

        # Always verify relevance of tool result vs user query
        vfy = relevance_verifier(user_input, tool_result)
        print(f'[Verifier] {vfy["verdict"]} (score={vfy["score"]})')

        # If tool result is relevant, feed it back into the model for a natural reply
        if vfy['verdict'] == 'relevant':
            followup = model_respond(
                f'The user asked: "{user_input}"\n'
                f'Tool output: {tool_result}\n'
                'Summarise the result in one short paragraph.',
                rag_context=''
            )
            print(f'Bot: {followup}')
        else:
            print('Bot: (Tool result was off-topic — please rephrase your query.)')

    else:
        # No tool matched → RAG-augmented LLM response
        hits = rag.retrieve(user_input, k=2)
        ctx = '\n'.join(h['chunk'] for h in hits if h['score'] > 0.3)
        response = model_respond(user_input, rag_context=ctx)

        # Verify the model response
        vfy = relevance_verifier(user_input, response)
        print(f'Bot: {response}')
        print(f'[Verifier] {vfy["verdict"]} (score={vfy["score"]})')

    print('-' * 50)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Save to your Google Drive folder
save_path = "/content/drive/My Drive/Colab_Models/Phi-3.5-COA-Finetune"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model saved to: {save_path}")

Mounted at /content/drive
Model saved to: /content/drive/My Drive/Colab_Models/Phi-3.5-COA-Finetune
